# PubMed RCT — Classification Run Analysis

Compares `test.csv` (gold labels) against `test_classified.csv` (model predictions)
for a single experiment run, using the standard multiclass metrics.

Set `RUN_DIR` below to point at the run you want to analyze (defaults to the most recent one under `runs/`).

In [ ]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

EXPERIMENT_DIR = Path.cwd()
RUNS_DIR = EXPERIMENT_DIR / "runs"

# Point this at a specific run, e.g. RUNS_DIR / "20260909-163215".
RUN_DIR = sorted(RUNS_DIR.iterdir())[-1]
print(f"Analyzing run: {RUN_DIR.name}")

In [ ]:
test = pd.read_csv(RUN_DIR / "test.csv")
classified = pd.read_csv(RUN_DIR / "test_classified.csv")
run_config = json.loads((RUN_DIR / "run_config.json").read_text())
category = run_config["category_name"]

assert len(test) == len(classified), "row count mismatch between test.csv and test_classified.csv"
assert test["label_gold"].equals(classified["label_gold"]), "gold labels don't line up between the two files"

print(f"Category: {category!r}  |  rows: {len(classified)}")
classified[["text", "label_gold", category]].head()

In [ ]:
def first_label(raw: str) -> str:
    labels = ast.literal_eval(raw)
    return labels[0] if labels else "none"


classified["_pred"] = classified[category].apply(first_label)
classified["_initial_pred"] = classified[f"{category}_initial"].apply(first_label)
classified["_n_labels"] = classified[category].apply(lambda raw: len(ast.literal_eval(raw)))

# "none" means the model reached consensus on "no fitting label" — drop it everywhere below.
n_unclassified = (classified["_pred"].str.lower() == "none").sum()
print(f"Dropping {n_unclassified} unclassified rows ({n_unclassified / len(classified):.1%}) from the analysis")
classified = classified[classified["_pred"].str.lower() != "none"].reset_index(drop=True)

n_ambiguous = (classified["_n_labels"] > 1).sum()
print(f"Rows with >1 predicted label: {n_ambiguous} ({n_ambiguous / len(classified):.1%}) — first label used below")

classified["_correct"] = classified["_pred"] == classified["label_gold"]
y_true = classified["label_gold"]
y_pred = classified["_pred"]
labels_order = sorted(set(y_true) | set(y_pred))

In [ ]:
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.3f}\n")
print(classification_report(y_true, y_pred, labels=labels_order, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels_order)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_order, yticklabels=labels_order)
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.title(f"Confusion Matrix — {category}")
plt.tight_layout()
plt.show()

## Critic / Debate Analysis

`--critics` mode samples each row `sampling_runs` times and votes; a row is only
escalated to a Critic (and, if it raises a real challenge, a Reconciler) when the
leading vote falls short of `consensus_threshold` and there's a genuine runner-up.
This looks at how accuracy differs across that split, and whether the critic step
nets out as a fix or a regression when it does change the label.

In [ ]:
votes = classified[f"{category}_votes"].apply(json.loads)
threshold = run_config["classifier_config"]["consensus_threshold"]
escalated = votes.apply(lambda v: max(v.values()) < threshold and len(v) >= 2)


def stage_group(esc: bool, challenged: bool, reconciled: bool) -> str:
    if not esc:
        return "Skipped critic (consensus)"
    if not challenged:
        return "Escalated, not challenged"
    if reconciled:
        return "Challenged & reconciled"
    return "Challenged, reconciler failed"


classified["_stage"] = [
    stage_group(e, c, r)
    for e, c, r in zip(escalated, classified[f"{category}_challenged"], classified[f"{category}_reconciled"])
]

print(f"Consensus threshold: {threshold}/{run_config['classifier_config']['sampling_runs']} votes\n")
print(f"Skipped critic loop:     n={(~escalated).sum():5d}  accuracy={classified.loc[~escalated, '_correct'].mean():.3f}")
print(f"Went through critic loop: n={escalated.sum():5d}  accuracy={classified.loc[escalated, '_correct'].mean():.3f}\n")

stage_summary = (
    classified.groupby("_stage")["_correct"]
    .agg(rows="size", accuracy="mean")
    .sort_values("accuracy")
)
stage_summary

In [ ]:
reconciled_mask = classified[f"{category}_reconciled"]
flipped = classified.loc[reconciled_mask, ["_initial_pred", "_pred", "label_gold"]].copy()
flipped["initial_correct"] = flipped["_initial_pred"] == flipped["label_gold"]
flipped["final_correct"] = flipped["_pred"] == flipped["label_gold"]

fixed = ((~flipped["initial_correct"]) & flipped["final_correct"]).sum()
broken = (flipped["initial_correct"] & (~flipped["final_correct"])).sum()
unchanged = len(flipped) - fixed - broken

print(f"Reconciled rows (label could have changed): {len(flipped)}")
print(f"  Fixed by critic  (incorrect -> correct): {fixed}")
print(f"  Broken by critic (correct -> incorrect): {broken}")
print(f"  Unchanged outcome:                       {unchanged}")
print(f"  Net effect:                              {fixed - broken:+d}\n")

pd.crosstab(
    flipped["initial_correct"].map({True: "Initial: correct", False: "Initial: incorrect"}),
    flipped["final_correct"].map({True: "Final: correct", False: "Final: incorrect"}),
    margins=True,
    margins_name="Total",
)

In [ ]:
plt.figure(figsize=(6, 3.5))
order = stage_summary.index
bar_color = sns.color_palette("Blues", 5)[3]
plt.barh(order, stage_summary["accuracy"], color=bar_color)
for i, (acc, n) in enumerate(zip(stage_summary["accuracy"], stage_summary["rows"])):
    plt.text(acc + 0.01, i, f"{acc:.3f} (n={n})", va="center")
plt.xlim(0, 1.05)
plt.xlabel("Accuracy")
plt.title("Accuracy by critic-pipeline stage")
plt.tight_layout()
plt.show()